# 06 · Access sensitivity runs (MLflow)

Four MLflow runs per job run (#33), so the team can see when a data refresh shifts the numbers:
- the 10, 15 and 20 min limits on the walking network (`gold.block_access`, `gold.access_sensitivity`)
- one step-free run: seniors at 1.0 m/s and 15 min on `silver.dist_matrix_step_free`

Each run logs the paces, the limit and the data download dates as params, and seniors cut off, centres excluded and the T4 count as metrics:
- `seniors_cut_off_summed_over_closures` (limit runs): each closable centre closing alone, added up, so a senior near several centres counts once per centre. Use it to compare runs, not as a headcount.
- `seniors_cut_off_worst_closure` (limit runs): the single closure that cuts off the most seniors.
- `seniors_cut_off_by_steps` (step-free run): seniors who can walk to a centre in 15 min today, but not without steps. A headcount.
- `seniors_without_walk`, `t4_blocks`, `centres_excluded` (every run): today, with every centre open. The app never reads MLflow. Overload sensitivity (eating share, distance decay) is Spec 2's.

In [0]:
%pip install "mlflow-skinny[databricks]>=2.15"

In [0]:
import json
import mlflow
from pyspark.sql import functions as F
from tiers import load_assumptions, load_paces, reach_m, tier_expr  # tiers.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("source_schema", "silver")  # dev_<name> for test runs
dbutils.widgets.text("target_schema", "gold")    # the gold schema 04_gold_access wrote; dev_<name>_gold for test runs
dbutils.widgets.text("experiment", "")           # default: hawkerreach_access_sensitivity in your workspace folder
dbutils.widgets.text("job_run_id", "")           # the job passes {{job.run_id}}, so the four runs can be grouped
SRC= f"{CATALOG}.{dbutils.widgets.get('source_schema')}"
GOLD= f"{CATALOG}.{dbutils.widgets.get('target_schema')}"
user= spark.sql("SELECT current_user()").first()[0]
EXPERIMENT= dbutils.widgets.get("experiment") or f"/Users/{user}/hawkerreach_access_sensitivity"
JOB_RUN_ID= dbutils.widgets.get("job_run_id") or "manual"
mlflow.set_experiment(EXPERIMENT)
print("reading", SRC, "and", GOLD, "| experiment", EXPERIMENT, "| job run", JOB_RUN_ID)

A= load_assumptions()
PACES= load_paces(A)                       # walk.pace.senior / typical / brisk
LIMIT_MIN= A["walk.limit_min"]["value"]    # 15
CUTOFF_M= A["walk.cutoff_m"]["value"]      # 2000
SENS_LIMITS= sorted({LIMIT_MIN, *(int(x) for x in str(A["access.sensitivity_limits_min"]["value"]).split(","))})

# data download dates: each bronze table's downloaded_at, and the OpenStreetMap file 03 saved.
# Always production's bronze, even on a test run: they say which raw files the numbers come from.
downloaded= {}
for t in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect():
    df= spark.table(f"{CATALOG}.bronze.{t.tableName}")
    if "downloaded_at" in df.columns:
        downloaded[f"downloaded.{t.tableName}"]= str(df.agg(F.max("downloaded_at")).first()[0])
osm= sorted(f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/bronze/raw_files") if f.name.startswith("osm_"))
downloaded["downloaded.osm"]= osm[-1].removesuffix(".osm.pbf").rsplit("_", 1)[-1] if osm else "none"

base_params= {**{f"pace.{k}": v for k, v in PACES.items()}, "cutoff_m": CUTOFF_M, "job_run_id": JOB_RUN_ID, **downloaded}
print(base_params)

In [0]:
blocks= spark.table(f"{GOLD}.block_access").select("block_id", "est_seniors", "tier_now", "nearest_walk_m")
centres= spark.table(f"{GOLD}.centres")

def centres_excluded(matrix):
    """Operating centres no HDB block can reach within the cutoff on this network: 04_gold_access's
    centre_excluded rule, applied here to the step-free network too."""
    return centres.join(spark.table(matrix).select("centre_id").distinct(), "centre_id", "left_anti").count()

def with_tier(walks, limit_min):
    """walks: block_id, est_seniors, walk_m (nearest; null if none in range). Adds the tier at this limit."""
    return walks.withColumn("tier", tier_expr(F.col("walk_m"), limit_min, PACES))

def today_counts(walks, limit_min):
    """T4 blocks, and seniors whose block isn't T1 (no centre a senior can walk to within the limit)."""
    r= with_tier(walks, limit_min).agg(
        F.sum(F.when(F.col("tier") == "T4", 1).otherwise(0)).alias("t4"),
        F.sum(F.when(F.col("tier") != "T1", F.col("est_seniors")).otherwise(0)).alias("seniors_without")).first()
    return {"t4_blocks": int(r["t4"] or 0), "seniors_without_walk": float(r["seniors_without"] or 0)}

def log_run(name, description, params, metrics):
    with mlflow.start_run(run_name=name, description=description):
        mlflow.set_tags({"job_run_id": JOB_RUN_ID, "notebook": "06_sensitivity_mlflow"})
        mlflow.log_params({**base_params, **params})
        mlflow.log_metrics(metrics)
    print(name, metrics)
    return metrics

results= {}

In [0]:
# 10, 15 and 20 min on the walking network
excluded= centres_excluded(f"{SRC}.dist_matrix")
sens= spark.table(f"{GOLD}.access_sensitivity")
walks= blocks.select("block_id", "est_seniors", F.col("nearest_walk_m").alias("walk_m"))
for limit in SENS_LIMITS:
    s= sens.filter(F.col("limit_min") == limit).agg(F.sum("seniors_cut_off").alias("summed"), F.max("seniors_cut_off").alias("worst")).first()
    results[f"limit_{limit}min"]= log_run(
        f"limit_{limit}min",
        f"Walking network at {limit} min. Seniors cut off are summed over every closable centre closing alone, "
        "so a senior near several centres counts once per centre: compare it across runs, don't read it as a headcount.",
        {"network": "walking", "limit_min": limit, "reach_m": [round(r) for r in reach_m(limit, PACES)]},
        {"seniors_cut_off_summed_over_closures": float(s["summed"] or 0),
         "seniors_cut_off_worst_closure": float(s["worst"] or 0),
         **today_counts(walks, limit),
         "centres_excluded": excluded})

In [0]:
# step-free: seniors at 1.0 m/s and the base limit when steps can't be used
step_free_nearest= spark.table(f"{SRC}.dist_matrix_step_free").groupBy("block_id").agg(F.min("walk_m").alias("walk_m"))
step_free_walks= blocks.join(step_free_nearest, "block_id", "left").select("block_id", "est_seniors", "tier_now", "walk_m")
# cut off by steps: a senior can walk to a centre within the limit today (T1), but not step-free
cut= with_tier(step_free_walks, LIMIT_MIN).filter((F.col("tier_now") == "T1") & (F.col("tier") != "T1"))
results["step_free"]= log_run(
    f"step_free_senior_{LIMIT_MIN}min",
    f"Step-free network (OpenStreetMap highway=steps removed) at {LIMIT_MIN} min and {PACES['senior']} m/s. "
    "Seniors cut off by steps is a headcount: T1 today, not T1 without steps.",
    {"network": "step_free", "limit_min": LIMIT_MIN, "pace": PACES["senior"]},
    {"seniors_cut_off_by_steps": float(cut.agg(F.sum("est_seniors")).first()[0] or 0),
     "blocks_cut_off_by_steps": cut.count(),
     **today_counts(step_free_walks.drop("tier_now"), LIMIT_MIN),
     "centres_excluded": centres_excluded(f"{SRC}.dist_matrix_step_free")})

In [0]:
# sanity checks: warnings, not failures, since every gold table is already written by now
base= results[f"limit_{LIMIT_MIN}min"]
if base["seniors_cut_off_summed_over_closures"] == 0:
    print("WARNING: no seniors cut off at the base limit; gold.access_sensitivity looks empty")
if results["step_free"]["seniors_without_walk"] < base["seniors_without_walk"] - 1:
    print("WARNING: fewer seniors lack a walk step-free than with steps; check the step-free snapping in 03_dist_matrix")
dbutils.notebook.exit(json.dumps({"experiment": EXPERIMENT, "job_run_id": JOB_RUN_ID, **results}))